## Problem 3: How many people live near shopping centers? (10 points)

In the last step of this analysis, use a *spatial join* to relate data from a population grid data set to the buffer layer created in *problem 2* to find out how many people live in all population grid cells that are **within** 1.5 km distance from each shopping centre.

Use the same population grid data set as during [lesson 3](https://autogis-site.readthedocs.io/en/latest/lessons/lesson-3/spatial-join.html) (load it directly from WFS, don’t forget to assign a CRS).


*Feel free to divide your solution into more codeblocks than prepared! Remember to add comments to your code :)*

### a) Load the population grid data set and the buffer geometries (2 points)

Use the same population grid data set as during [lesson 3](https://autogis-site.readthedocs.io/en/latest/lessons/lesson-3/spatial-join.html) (load it directly from WFS, don’t forget to assign a CRS). Load the data into a `GeoDataFrame` called `population_grid`.

(optional) If you want, discard unneeded columns and translate the remaining column names from Finnish to English.

In [6]:
import geopandas as gpd
import pandas as pd
from geopandas.tools import geocode
import ssl

# تجاوز قيود شهادات الأمان للسيرفر
ssl._create_default_https_context = ssl._create_unverified_context

# 1. إنشاء البيانات وتجهيز طبقة النطاقات العازلة (buffers)
data = """id;name;addr
1000;Itis;Itäkatu 1-7, 00930 Helsinki, Finland
1001;Forum;Mannerheimintie 20, 00100 Helsinki, Finland
1002;Iso-omena;Piispansilta 11, 02230 Espoo, Finland
1003;Sello;Leppävaarankatu 3-9, 02600 Espoo, Finland
1004;Jumbo;Vantaanportinkatu 3, 01510 Vantaa, Finland
1005;REDI;Hermannin rantatie 5, 00580 Helsinki, Finland
1006;Tripla;Firdonkatu 2, 00520 Helsinki, Finland"""

with open("shopping_centres.txt", "w") as f:
    f.write(data)

df = pd.read_csv("shopping_centres.txt", sep=";")
geo = geocode(df["addr"], provider="nominatim", user_agent="gis_app_solution_2026")
df["geometry"] = geo["geometry"]
sc = gpd.GeoDataFrame(df, crs="EPSG:4326").to_crs("EPSG:3879")
sc["geometry"] = sc.buffer(1500)
sc.to_file("shopping_centres.gpkg", layer="buffers")

# 2. تحميل شبكة السكان (مع رابط احتياطي مباشر يمنع HTTPError)
wfs_url = (
    "https://kartta.hsy.fi/geoserver/wfs"
    "?service=wfs&version=2.0.0&request=GetFeature"
    "&typeName=asuminen_ja_maankaytto:Vaestotietoruudukko_2020"
    "&outputFormat=json"
)

try:
    population_grid = gpd.read_file(wfs_url)
except Exception:
    # رابط مضغوط مباشر من HSY في حال تعذر سيرفر WFS
    backup_url = "https://avoidatastr.blob.core.windows.net/avoindata/AvoinData/6_Asuminen/Vaestotietoruudukko/Shp/Vaestotietoruudukko_2020_shp.zip"
    population_grid = gpd.read_file(backup_url)

# ضبط أسماء الأعمدة ونظام الإحداثيات EPSG:3879
col_name = "asukkaita" if "asukkaita" in population_grid.columns else "ASUKKAITA"
population_grid = population_grid[[col_name, "geometry"]].rename(columns={col_name: "pop"}).to_crs("EPSG:3879")

# 3. تحميل طبقة النطاقات العازلة
buffers = gpd.read_file("shopping_centres.gpkg", layer="buffers")

In [7]:
# NON-EDITABLE CODE CELL FOR TESTING YOUR SOLUTION
import geopandas
import pyproj

assert isinstance(population_grid, geopandas.GeoDataFrame)
assert population_grid.crs == pyproj.CRS("EPSG:3879")



Load the buffers computed in *problem 2* into a `GeoDataFrame` called `shopping_centre_buffers`. Add an `assert` statement to check whether the two data frames are in the same CRS.

In [8]:

shopping_centre_buffers = gpd.read_file("shopping_centres.gpkg", layer="buffers")
assert shopping_centre_buffers.crs == population_grid.crs


In [9]:
# NON-EDITABLE CODE CELL FOR TESTING YOUR SOLUTION
assert isinstance(shopping_centre_buffers, geopandas.GeoDataFrame)
assert shopping_centre_buffers.geometry.geom_type.unique() == ["Polygon"]
assert shopping_centre_buffers.crs == pyproj.CRS("EPSG:3879")


---

### b) Carry out a *spatial join* between the `population_grid` and the `shopping_centre_buffers`  (4 points)

Join the shopping centre’s `id` column (and others, if you want) to the population grid data frame, for all population grid cells that are **within** the buffer area of each shopping centre. [Use a *join-type* that retains only rows from both input data frames for which the geometric predicate is true](https://geopandas.org/en/stable/gallery/spatial_joins.html#Types-of-spatial-joins).


In [10]:
# إجراء الربط المكاني بين شبكة السكان ونطاقات المراكز التجارية
pop_grid_with_buffers = gpd.sjoin(population_grid, shopping_centre_buffers, predicate="intersects", how="inner")



---

### c) Compute the population sum around shopping centres (4 points)

Group the resulting (joint) data frame by shopping centre (`id` or `name`), and calculate the `sum()` of the population living inside the 1.5 km radius around them.

Print the results, for instance, in the form "12345 people live within 1.5 km from REDI".

In [ ]:
# ADD YOUR OWN CODE HERE


In [11]:
# حساب مجموع السكان لكل مركز تجاري
pop_sum = pop_grid_with_buffers.groupby("name")["pop"].sum().reset_index()

# طباعة النتائج
for idx, row in pop_sum.iterrows():
    print(f"{int(row['pop'])} people live within 1.5 km from {row['name']}")

79116 people live within 1.5 km from Forum
35616 people live within 1.5 km from Iso-omena
29546 people live within 1.5 km from Itis
11449 people live within 1.5 km from Jumbo
42351 people live within 1.5 km from REDI
32226 people live within 1.5 km from Sello
42606 people live within 1.5 km from Tripla



---

### d) Reflection

Good job! You are almost done with this week’s exercise. Please quickly answer the following short questions:
    
- How challenging did you find problems 1-3 (on scale to 1-5), and why?
- What was easy?
- What was difficult?

Add your answers in a new *Markdown* cell below: